# Train an RNN with truncated backprop through time

Long sequences don't fit in one backward pass. Truncated BPTT cuts the sequence into chunks, carries the hidden state **forward** across chunks but stops gradients at chunk boundaries with `detach()`.

Write `tbptt_train(x, y, W_ih, W_hh, b_h, W_out, b_out, h0, chunk, lr)`:

- `x`: float64 `(T, B, D)` inputs, `y`: float64 `(T, B, K)` targets (time first).
- Cell (one bias, `nn.RNNCell` weight layout): $h_t = \tanh(x_t W_{ih}^\top + h_{t-1} W_{hh}^\top + b_h)$, with `W_ih` `(H, D)`, `W_hh` `(H, H)`, `b_h` `(H,)`.
- Readout: $\hat y_t = h_t W_{out}^\top + b_{out}$, with `W_out` `(K, H)`, `b_out` `(K,)`.
- `h0`: `(B, H)` initial state. `chunk`: int $\ge 1$. `lr`: float.

## The loop
Split time into consecutive chunks `[0, chunk), [chunk, 2*chunk), ...`; the last one may be shorter. For each chunk:

1. Start from the hidden state carried over from the previous chunk, **detached** (the first chunk starts from `h0`).
2. Run the cell over the chunk and compute the loss: mean squared error over all `len * B * K` entries of the chunk.
3. Zero the gradients, backward, and apply one plain SGD step $\theta \leftarrow \theta - \eta \nabla\theta$ to all five parameters.
4. Carry the chunk's last hidden state to the next chunk. It was computed with the weights from before this step; don't recompute it.

## Return
A tuple `(losses, W_ih, W_hh, b_h, W_out, b_out, h_last)`: `losses` is float64 `(n_chunks,)` with each chunk's loss before its update, then the five trained parameters and the final hidden state `(B, H)`. Don't modify the inputs.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def tbptt_train(x, y, W_ih, W_hh, b_h, W_out, b_out, h0, chunk, lr):
    """Return (losses, W_ih, W_hh, b_h, W_out, b_out, h_last)."""
    params = [p.clone().requires_grad_(True) for p in (W_ih, W_hh, b_h, W_out, b_out)]
    Wih, Whh, bh, Wo, bo = params
    h = h0
    losses = []
    for s in range(0, x.shape[0], chunk):
        # TODO: run the cell over x[s:s+chunk], MSE loss, backward, SGD step, carry h
        pass
    return (torch.tensor(losses, dtype=torch.float64), *[p.detach() for p in params], h)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0]], [[0.0]], [[-1.0]]], "dtype": "float64"}, {"__nd__": [[[0.5]], [[0.5]], [[0.0]]], "dtype": "float64"}, {"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [[0.5]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, 2, 0.1], "e": {"__tup__": [{"__nd__": [0.043545601647692875, 0.4466797158959072], "dtype": "float64"}, {"__nd__": [[0.9182022445721982]], "dtype": "float64"}, {"__nd__": [[0.5356671324166253]], "dtype": "float64"}, {"__nd__": [0.07666056965507055], "dtype": "float64"}, {"__nd__": [[0.8960447416143891]], "dtype": "float64"}, {"__nd__": [0.12116883949888027], "dtype": "float64"}, {"__nd__": [[-0.6658012428260026]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0, 1.0]], [[1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[0.0]], [[0.0]]], "dtype": "float64"}, {"__nd__": [[0.5, -0.5], [1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.5, 0.0], [0.0, 0.5]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.1], "dtype": "float64"}, {"__nd__": [[0.2, -0.2]], "dtype": "float64"}, 1, 0.5], "e": {"__tup__": [{"__nd__": [0.838993465522742, 2.8495384751920705], "dtype": "float64"}, {"__nd__": [[-0.37055663943332984, -1.37055663943333], [1.1312494530724322, 0.1312494530724322]], "dtype": "float64"}, {"__nd__": [[0.32224558821333366, 0.20738236643561747], [-0.031666672688479815, 1.0026822622452318]], "dtype": "float64"}, {"__nd__": [-0.8705566394333298, 0.1312494530724322], "dtype": "float64"}, {"__nd__": [[-0.7592513494986027, 0.47052028953584296]], "dtype": "float64"}, {"__nd__": [0.8720917394040479], "dtype": "float64"}, {"__nd__": [[-0.9880935724166022, 0.07501206556406276]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.8019314252534474, -1.324358995628145, -0.24836162209524854], [0.4204452380655215, 1.1360465324896427, 0.10970639932180819]], [[-0.5526473205362324, -0.7847803553442784, 0.7487457707345911], [1.6347830429585775, 0.27276877584472176, -1.2333286640307717]], [[-0.9582652054360887, 1.6000190889991115, 0.2028824405086084], [-1.7321348424395848, -0.08369619281702581, -1.1632259734447485]], [[-0.6292880940615545, -0.48800582327685743, -0.7133133716322436], [0.5533784703532895, -0.06308597192528916, -0.5894312580326048]], [[0.40963782655711695, 0.8298553070613239, -1.643023371405677], [-0.256730126365494, -0.9807473560440125, -0.17315522486203205]], [[-1.2894187467538587, 0.0206903940375912, -0.03788574104406823], [-0.304337750958489, -1.0479265051202462, -0.3961903304730927]], [[-1.091328901695709, -1.3552087462047395, 0.22478573245989314], [-1.109349937891366, 1.1702961011782933, 0.7165876558738361]], [[-1.9978166924497212, 0.272128869412488, -1.1017166275810448], [0.033057220158269195, 0.04363199256942161, -1.9884297882311208]]], "dtype": "float64"}, {"__nd__": [[[-0.7187003889821505], [0.4081669545829814]], [[-0.9767159023094614], [0.8849396340494994]], [[-0.7370863231807921], [0.31750145230146615]], [[-0.1981406621448166], [0.7684861594525221]], [[-0.5721282613013278], [0.5808249835674885]], [[0.6290420575476957], [0.310200597725079]], [[0.9798967789549838], [-0.7131250427041856]], [[-0.5874249563474043], [-0.6895653246849386]]], "dtype": "float64"}, {"__nd__": [[-0.11671126188288501, -0.1278950156996955, 0.4810002659215472], [-0.5907234039781079, 0.36902094892284204, -0.5494863815182032], [-0.16564544634995837, -0.42023658421110555, 0.724365644460836], [0.28410654989414663, 1.2158662514226062, 0.32095818954116023]], "dtype": "float64"}, {"__nd__": [[0.21124818342979385, 0.21017071906633503, -0.1516528839773879, -0.01750711165959552], [0.3375972169361565, -0.0991376912932429, 0.047199882822774666, -0.005305865104322449], [0.15230412320818518, -0.09122718548684315, -0.03809047218921207, 0.06059535716803761], [0.025755784621701923, -0.21624318657045502, 0.22394576079736095, -0.324620302061728]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-0.6005577744017633, -0.641245897370231, 0.4834861394666074, -0.180304184449635]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 3, 0.1], "e": {"__tup__": [{"__nd__": [0.7202943504819319, 0.2738695247685008, 0.3129682083476634], "dtype": "float64"}, {"__nd__": [[-0.12519475152716822, -0.15579149976513013, 0.5154437740912525], [-0.6063686010618795, 0.3550253012683892, -0.5188980861921106], [-0.1786778829979346, -0.4084638681014806, 0.7067912749829084], [0.2804698309562356, 1.2136134304819381, 0.323204635859033]], "dtype": "float64"}, {"__nd__": [[0.24457947150071677, 0.18580617704027041, -0.09366404647735264, -0.01771608148550392], [0.35904750953987846, -0.12450637538950517, 0.09354851178184997, -0.01161133305981684], [0.13449645646634922, -0.08000943960196395, -0.06779795934839355, 0.06495082162894152], [0.03081102468403452, -0.2159012720571742, 0.23234706352345483, -0.3329901211594667]], "dtype": "float64"}, {"__nd__": [-0.0600666469731764, -0.05463520688165767, 0.03430523500246969, 0.001461413732938376], "dtype": "float64"}, {"__nd__": [[-0.5964513473135598, -0.6276235679314387, 0.451641930029522, -0.2035624322021402]], "dtype": "float64"}, {"__nd__": [0.08286207428104789], "dtype": "float64"}, {"__nd__": [[-0.2880595965526669, 0.9727254254722111, -0.5051037266589867, -0.14303178502769479], [-0.7562353556771567, 0.7488045208367992, -0.8711008356017361, -0.7537642573190007]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.9710363785210655, -1.1360213941896466]], [[0.42113113746240616, -1.054840662577835]], [[-1.2720782100976422, 0.6139930624688609]], [[-1.1967077271925706, -0.32243815075139726]], [[-0.006761540346521785, -0.4453353669298123]], [[-0.05409396159544828, 1.3387734990547715]]], "dtype": "float64"}, {"__nd__": [[[-0.8254711345288706, -0.8254711345288706]], [[-0.5226064422629313, -0.5226064422629313]], [[-0.9686180397857508, -0.9686180397857508]], [[-0.12259230902129915, -0.12259230902129915]], [[-0.11587901897309318, -0.11587901897309318]], [[-0.06200617120148607, -0.06200617120148607]]], "dtype": "float64"}, {"__nd__": [[-0.2584470762946375, -0.6296535794391138], [-0.9183728525968398, -0.10238305841693353], [-0.17612850520561524, 0.13254545346516175]], "dtype": "float64"}, {"__nd__": [[-0.13401586997586395, -0.13817102274399962, -0.20822592059555406], [-0.15004172224593068, 0.04625346601186602, -0.10979837472233435], [0.028995065385376204, 0.5488254708595359, 0.13830949666812553]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-0.7880449227176131, 0.8667647928065534, 0.17390518499426771], [-0.4707066432254923, 0.4535244788429051, 0.008827805454919485]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0]], "dtype": "float64"}, 6, 0.2], "e": {"__tup__": [{"__nd__": [0.6178593969070407], "dtype": "float64"}, {"__nd__": [[-0.3821473795374065, -0.6054891779780074], [-0.857327353580115, -0.1008146097311026], [-0.1549510253842461, 0.1389086783639109]], "dtype": "float64"}, {"__nd__": [[-0.11213353613117708, -0.14015107661422133, -0.1820348251215749], [-0.17749823850465668, 0.02823370868738931, -0.13343578682971527], [0.015827990794943146, 0.5417000086628129, 0.13259845312404722]], "dtype": "float64"}, {"__nd__": [0.11807195200633827, -0.10817144573870847, -0.03715468979341809], "dtype": "float64"}, {"__nd__": [[-0.7991502424621133, 0.7843374225356183, 0.14585310328975767], [-0.4846145334581099, 0.39275901872018226, -0.010596478631251599]], "dtype": "float64"}, {"__nd__": [-0.1336214051866059, -0.10584299955907549], "dtype": "float64"}, {"__nd__": [[-0.7255005786699975, -0.13752744568008032, 0.23081326513035316]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.6152185330350691, -0.6335089254336164], [-0.9934317902518573, 0.048118983989838576], [1.0688166937495658, -0.3250520988063787]], [[0.42082412505635486, 1.875646053775279], [-1.2145906329798497, 0.2575223239234917], [-0.30643692685187435, -1.0592560998278446]], [[-1.0258471994365972, -0.0152851431653622], [0.4338634112001693, -0.531589820743032], [0.05439054435949247, 0.7222086544347099]], [[0.2965612469178253, 0.8686701897408925], [0.4613956081509272, 0.4880056756807095], [1.8266340649418156, 0.6231718736925668]], [[0.12947113235560923, 0.7099438034419987], [-0.9191236755082103, -0.3377047857608698], [0.793220854194603, 0.6307867000938839]], [[1.5483616908312823, 0.010473028918682835], [-1.4623277904204515, 1.9472469308762064], [1.0928928105587499, -1.0587374336035666]], [[1.3758236699684163, 0.03240279877908362], [-1.813928769904867, -0.4203539214229468], [-0.5089038339937678, 1.590848160322242]]], "dtype": "float64"}, {"__nd__": [[[-0.5771370004874774], [-0.8379040404081122], [0.8766317562720696]], [[-0.1931723848264234], [-0.8037492324452344], [0.690644433446815]], [[-0.9391823562692501], [-0.9793929755539608], [0.7289386581567922]], [[-0.7978258633489076], [-0.9668938054008511], [0.47783481648685455]], [[-0.7133096163104625], [-0.7893230028901902], [-0.2907710195250381]], [[0.6846711852981692], [0.5249209329956724], [-0.9833256558709684]], [[0.8476905017877532], [0.6997458428529307], [-0.7701150916714167]]], "dtype": "float64"}, {"__nd__": [[-0.3960343477097619, -0.12680867018968403], [-0.1383804034226857, -0.1894863814886314], [-0.45703703051630923, 0.10954345451283838], [0.5384023232067773, 0.31198882558587604], [-0.4637495580296551, -0.5749078785808449]], "dtype": "float64"}, {"__nd__": [[0.02660329533341744, -0.15799612774156727, -0.14090932279122986, -0.3755911259724487, 0.4361431813179633], [0.20496227345596776, -0.2177722997929449, 0.20308198944158265, 0.30018924275408104, -0.5343203725739287], [-0.12274778564157417, -0.08675344891782803, 0.14495410624057192, -0.027160494118035047, -0.051526013111187284], [-0.013052239390642581, 0.41441622671826317, 0.4829863160497801, -0.11735475895515468, -0.20711445326588257], [0.6020731870588022, -0.21907756004095572, -0.12821658904425123, 0.008179874162332257, 0.10805714517802024]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.5145177747189055, 0.19548861083506977, -0.43619153584759596, 0.2531911631748551, 0.12495878992561109]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 1, 0.05], "e": {"__tup__": [{"__nd__": [0.5572002413974667, 0.40383938371888717, 0.8279846341405103, 0.5548298124399145, 0.13997001052174804, 0.7558174668025718, 0.7059950002477061], "dtype": "float64"}, {"__nd__": [[-0.3895017861449046, -0.12772712296894884], [-0.12109839427835625, -0.18327477023223537], [-0.48529898959000245, 0.11240831229217967], [0.5420264433091582, 0.3261498952948361], [-0.45866749610485175, -0.5705257292048812]], "dtype": "float64"}, {"__nd__": [[0.018457099733609053, -0.15740679833325302, -0.14433290881984365, -0.37663491028228624, 0.42502428254682334], [0.19786741643488978, -0.215927626852303, 0.1991601374469628, 0.30276507403479774, -0.5409559599770348], [-0.10670855070446987, -0.088998739552753, 0.1581542567684352, -0.03283785301711621, -0.036599054833571294], [-0.01906762847872131, 0.4169649151060659, 0.4801651861162078, -0.11952140916631192, -0.21119845794265688], [0.6001830377537124, -0.21681062978255683, -0.129779436803473, 0.006531464922905344, 0.10865880999847358]], "dtype": "float64"}, {"__nd__": [-0.03927481105641051, -0.01331229300824777, 0.04219927646073613, -0.019205674357493263, -0.007082860625588426], "dtype": "float64"}, {"__nd__": [[0.47879622555132195, 0.19366343990549037, -0.4554808495630153, 0.28580098311953855, 0.06548026467634191]], "dtype": "float64"}, {"__nd__": [-0.05782450459855939], "dtype": "float64"}, {"__nd__": [[-0.8171961268123018, 0.28220550608404504, -0.5191265489979663, 0.41893516985641427, -0.7863105182724766], [0.44033766773350136, 0.7004515999608233, 0.7710809771208291, -0.7437477794348134, 0.8741291400137405], [-0.29853622975838157, -0.2620548267433153, 0.40399460035233364, 0.19487230666102556, -0.8124724859010389]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.876918992454176, -0.01499979815571943, -1.3369728999172332, -1.0450199143839616], [1.450153058953162, -0.5401312070783946, -2.1044660100855603, -0.5806997054894565]], [[1.5099831293121058e-05, 1.1888306785373703, -1.014468137602427, 0.6666833259020761], [0.7952990996016167, -0.6993883083236738, -0.18758970531896946, 1.7694502363979239]], [[1.720484746826155, 0.8555220049018919, 0.33194635951240653, 1.1383096209632015], [-0.14067728561527593, -0.09509306569845781, -0.8603601395273159, 0.005563384668739004]], [[-0.0820166249133826, 2.7736316673693673, -0.19284316578043667, 1.2708179900134384], [1.3204972743933365, -0.18647870191497867, 1.1693971923001296, -2.1768136494850046]], [[0.09492718485511631, 0.8572613994662929, -2.39786527267812, -1.159171198248098], [1.0560736173734917, -0.25701076898467345, -1.0981986938209587, -0.3735956685207676]], [[-0.5425883064344673, 0.7244473133352144, 0.4538858794596457, -0.2802283393557666], [-0.6653426856286421, -0.0538756638943387, 1.325155927345223, 0.3490261125780868]], [[0.6389455936428303, -0.1506815972085555, -1.2918162500762125, -0.7108435355871252], [0.6512365658972803, 0.10321610258141592, -0.9038398688268016, -1.617870425602848]], [[0.11068190035142705, -0.339531008698164, -0.350530218612727, -0.3294499574610804], [0.28390511348310316, 0.29444772495280114, 0.9621285502204014, 0.9284215804413863]], [[1.3320930804934341, 0.8063630567820633, -0.3356080333276904, -0.05426894624359061], [0.4942150948314312, -0.31029227000142345, 0.5494387802798892, 1.1536094769949883]], [[-0.8012460099660716, -1.770384050846499, 1.1936549552915572, 0.01775991033501458], [0.7086698231113021, -1.3323053543858547, 1.7329925604781644, 0.7386198894684642]]], "dtype": "float64"}, {"__nd__": [[[0.9535092226991408, 0.9535092226991408, 0.9535092226991408], [0.992731423437084, 0.992731423437084, 0.992731423437084]], [[0.9535046720473427, 0.9535046720473427, 0.9535046720473427], [0.7809219746998911, 0.7809219746998911, 0.7809219746998911]], [[-0.44020429399309413, -0.44020429399309413, -0.44020429399309413], [0.8607889594912262, 0.8607889594912262, 0.8607889594912262]], [[-0.36516455688963156, -0.36516455688963156, -0.36516455688963156], [-0.27988996731443133, -0.27988996731443133, -0.27988996731443133]], [[-0.4517596466333762, -0.4517596466333762, -0.4517596466333762], [-0.9734280355016373, -0.9734280355016373, -0.9734280355016373]], [[0.0737844475535347, 0.0737844475535347, 0.0737844475535347], [-0.6244369232279962, -0.6244369232279962, -0.6244369232279962]], [[-0.5354952325667319, -0.5354952325667319, -0.5354952325667319], [-0.9701010919070797, -0.9701010919070797, -0.9701010919070797]], [[-0.6255029048552623, -0.6255029048552623, -0.6255029048552623], [-0.9992490888297173, -0.9992490888297173, -0.9992490888297173]], [[-0.9059946432114838, -0.9059946432114838, -0.9059946432114838], [-0.8613013898163518, -0.8613013898163518, -0.8613013898163518]], [[-0.9344185236283313, -0.9344185236283313, -0.9344185236283313], [-0.3232441039381916, -0.3232441039381916, -0.3232441039381916]]], "dtype": "float64"}, {"__nd__": [[-0.06303708684573414, -0.1759940194025436, 0.05694794022762152, -0.08336567028267486], [-0.1393450636219276, -0.5597999877627873, 0.6314047194605225, -0.019788842284779562], [0.037317818694801407, 0.04645799430023501, -1.1791086683709355, 0.2635707290621626], [-0.16457746531267947, -0.14461423229617304, 0.03489393628740623, -0.48623780266421046], [0.42169040499385363, -0.3479323119035048, -0.45182902735643965, -0.43109480049118726], [0.07937634756996338, 0.3728897371812343, -0.29752661777313055, -0.7550933379576436]], "dtype": "float64"}, {"__nd__": [[0.21688000455686163, -0.21740927843878027, 0.24389372767332285, -0.011083099477571863, -0.14255007519047683, -0.1306476075316172], [0.09427803791652993, 0.18154225744711028, -0.04045132831543485, 0.19329217139233576, 0.4260026427574846, -0.04252076292087779], [-0.23462981313090447, -0.2152285698760291, -0.2838010159253463, -0.13057116340762984, -0.1477408180753344, -0.2704855843692232], [0.10822340780882184, -0.5330373423009529, -0.2696069611200705, 0.41136155236439, 0.2118405336297177, 0.10671440549679992], [0.09333761455881695, 0.3614910366573517, 0.10933494166764123, 0.015835356619837888, -0.2511053913185252, -0.015037176119224204], [-0.1498034438554052, -0.14414742437350378, -0.1457934117581913, -0.009113503092637762, -0.06730193925357059, 0.177814688447041]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-0.6630907209558661, -0.08473509837143318, 0.9297487482172506, 0.01944866859748845, 0.599307610724321, 0.5684317486016571], [0.6583239101686803, 0.3690595847626233, -0.6902230156264456, -0.13369385289350136, -0.19016779123431019, -0.7744276521667103], [-0.23194081058756896, -0.6250399175668997, -0.8327757230096916, 0.10544812828072586, -0.021795636222830456, -0.3439074547875926]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 4, 0.1], "e": {"__tup__": [{"__nd__": [1.5872140510201949, 1.828330024490657, 1.527542454695695], "dtype": "float64"}, {"__nd__": [[0.029398520066091532, -0.05934735793941903, -0.19919664919719604, -0.2041015446126952], [-0.11576032155086649, -0.5547710695203203, 0.6263106752100681, -0.0256860153701563], [0.01649736503600606, 0.0013249593600276957, -1.071498217023187, 0.34285863392389593], [-0.17528380372781802, -0.1569491099281005, 0.0684256288828823, -0.47158347602308], [0.4095578095270644, -0.3882382323074253, -0.38729520513001847, -0.4003812588188666], [0.034950353162324574, 0.3074670046777063, -0.13780651483830025, -0.7212102528225081]], "dtype": "float64"}, {"__nd__": [[0.2296711392530333, -0.20015621507620288, 0.2026783453535037, 0.013965743557545568, -0.19371314179647114, -0.12431453320328821], [0.09698792295846255, 0.19415326655653825, -0.05867038807667804, 0.20333200078850822, 0.4153313402973625, -0.046332755967627585], [-0.2457819020233904, -0.25879000651098827, -0.23271514269688748, -0.1491547299444779, -0.1318258208784573, -0.2591499221862138], [0.1073641297914938, -0.5324963539139864, -0.2671706678078653, 0.41157432578782455, 0.2185977103817395, 0.10679528083096931], [0.08842348840523734, 0.35182050166508627, 0.1261133662940454, 0.0029512072699599094, -0.23549514325026297, -0.017830717920657598], [-0.1553534119291405, -0.16376766769522194, -0.12003065817086303, -0.0011850551714830421, -0.03427392674170407, 0.20099500852034424]], "dtype": "float64"}, {"__nd__": [0.009140005561285136, 0.023090958815113055, 0.004925840453649969, -0.002381980617396421, 0.005413166369814293, 0.007747765117386899], "dtype": "float64"}, {"__nd__": [[-0.6550955559142285, 0.03436099625674386, 0.7955117489185846, 0.004808342192088517, 0.4759840662831767, 0.45264936042535797], [0.6470042042440078, 0.1952756312863944, -0.5006505546568352, -0.13298326174955788, -0.08283936275613422, -0.623969988731447], [-0.23524395361816464, -0.6872646954823622, -0.7499206330585546, 0.09640528306447516, 0.018339508306344343, -0.2939961606150492]], "dtype": "float64"}, {"__nd__": [-0.07734358849553391, -0.057229680606972214, -0.06290804481464247], "dtype": "float64"}, {"__nd__": [[0.17229518336649338, 0.9555191997899392, -0.8955177116586847, 0.6614983375994055, -0.455843306914982, -0.6548085327043897], [0.07819330947297529, 0.9209329175019808, -0.8972686011044879, -0.6335784409506374, -0.1581439875683855, -0.9066939664523519]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.9505863626128569], [-0.541273442072544]], [[-0.39409036937385966], [-0.4081102825826052]], [[-0.052707747928468524], [-0.43554067396098334]], [[0.3869719823711992], [0.5228708948391112]], [[0.35786967764803124], [-0.6631035145259176]]], "dtype": "float64"}, {"__nd__": [[[-0.8137564421513132], [-0.515227816803271]], [[-0.9745437074479092], [-0.8130568733998236]], [[-0.985001807317371], [-0.9827754888416668]], [[-0.8470511638070963], [-0.7591807468946418]], [[-0.6072087447300136], [-0.9989587075253883]]], "dtype": "float64"}, {"__nd__": [[-0.3082530733410392], [0.2174320517354771]], "dtype": "float64"}, {"__nd__": [[-0.06966514242231915, -0.015256432887041428], [-0.8810533700898602, -0.9546309170431714]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.16857409603437307, 0.16477932817537805]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, 2, 0.0], "e": {"__tup__": [{"__nd__": [0.6397614791975412, 0.8038456697186864, 0.7111624722077081], "dtype": "float64"}, {"__nd__": [[-0.3082530733410392], [0.2174320517354771]], "dtype": "float64"}, {"__nd__": [[-0.06966514242231915, -0.015256432887041428], [-0.8810533700898602, -0.9546309170431714]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.16857409603437307, 0.16477932817537805]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": [[-0.10227502999053267, 0.14024597280888748], [0.21162387964865328, -0.07079930570290127]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.3934318714435818, 0.7211631188958838, 0.22173880016804176], [0.01636245608868217, -1.426740428209398, 0.8831758896325703], [-0.6035860153728927, 0.5024844545833684, 0.20004164543806374], [0.272097020758701, 0.8926752155435728, -0.48986611197868063]], [[-0.3872351816195507, -0.8558614733519333, -0.37715202345150317], [1.6659407140512403, -0.3633186666203786, 1.1856163121936107], [0.8260114337421408, 0.7292817915893176, 0.8030151370721609], [-1.0142765548969686, 0.37344767888543307, 1.588412097484749]], [[-0.6957959363309963, -0.0471070855877405, -0.6891591571615298], [0.31701151121305016, -1.0785301318658582, 1.6096797942508279], [-0.37683478895943645, -1.542661572000243, 1.790790910814277], [0.970522894800899, 0.518511481410699, 0.5325875434742989]], [[-0.3577980078127549, -0.22329559977220623, 0.7240192298296324], [0.9176249577098493, 0.9326124031440995, 0.2125509856107767], [-0.466388093105908, -0.19208732917753973, 1.1095997115146592], [1.4399181149572342, -0.34131239008610353, -0.47730400254445376]], [[1.2475496699476993, 0.7236490231334424, 0.6972581900253083], [-1.3123336629533346, -0.3737574414663936, -0.5149955292041763], [0.20790465904469757, -1.6849419797309495, 1.2717528081338776], [0.30013179535055773, 0.43483702547979786, -1.1841403768848728]], [[-0.026420371368168806, 1.3346502881568998, -1.5696956024589206], [0.3057680573775562, 0.6988049143143882, -0.6919471030872025], [0.8147412282685731, -0.33006903056415593, 1.2297665530315567], [0.5275117754251525, 0.22436588572838914, 2.008915981414082]], [[0.8318964583740944, 1.3086357086483342, -1.1895196722305454], [-0.6466070677186754, 0.8365690029317673, 0.9056118633317605], [-0.05574002999013012, 0.7797265342256705, -1.0583537461465884], [0.21383522572037922, -1.1472838649378465, 0.3405705337781045]], [[-0.4861719071399926, -0.6409975166442754, 0.05629903018263689], [1.8174600636946423, -1.011761734455417, -0.6017506161418752], [1.067664145735261, -0.4235666699986711, 1.7692642772096374], [0.22252970384637974, 0.09063191415105347, 0.35688413886287884]], [[1.741179596515403, 0.6553615344906699, 0.5167121329088065], [0.2518577631500006, -1.562418833300465, -0.8852720281451512], [1.4833441210372633, 0.8218783319217637, 0.5060648618402573], [0.6370627277466291, -0.5435930521122606, 0.33796426767211185]], [[-0.900794630349013, 0.39136232136016047, -0.7141351107069186], [-0.21381982915618122, 0.4534148448314499, -0.32678770337045643], [0.6475819228802786, 0.45603827622850734, 0.35945813704442325], [1.3244113828426742, -1.4738846764712352, -1.5962822788262483]], [[0.6162417919107772, 0.9407940950142847, 2.0116404277145703], [0.7562567989915012, 0.8302149983209312, -0.8830800515074957], [0.7526217774690592, 0.6738389380752741, -1.0558651891629185], [-0.896317509048086, -0.11103120279927593, 0.2882537511901322]], [[-2.944785538646996, 0.0703055101868928, 0.23085449610631834], [-0.17683635976985834, 0.26682489806649157, 0.4559062316030521], [-1.011638037208477, 0.4161801684774495, -0.25130719973217774], [-0.9458787787894589, 0.20914079384282933, 1.487644239755298]]], "dtype": "float64"}, {"__nd__": [[[0.9843121157418864, 0.9843121157418864], [0.016361725978478268, 0.016361725978478268], [-0.5675985027467751, -0.5675985027467751], [0.26875189630887814, 0.26875189630887814]], [[0.8448028934257844, 0.8448028934257844], [0.9937895508906038, 0.9937895508906038], [0.22059593847591444, 0.22059593847591444], [-0.675895826230051, -0.675895826230051]], [[0.3054402629411727, 0.3054402629411727], [0.9095824064592892, 0.9095824064592892], [-0.15379652373054753, -0.15379652373054753], [0.22636419738472616, 0.22636419738472616]], [[-0.04737950999323228, -0.04737950999323228], [0.2227681093133036, 0.2227681093133036], [-0.5816840142365708, -0.5816840142365708], [0.9952540311772786, 0.9952540311772786]], [[0.9320943041258363, 0.9320943041258363], [0.999428508248713, 0.999428508248713], [-0.40126070782250595, -0.40126070782250595], [0.9219941277041502, 0.9219941277041502]], [[0.9222002455391877, 0.9222002455391877], [0.9428954108299364, 0.9428954108299364], [0.3911201869765686, 0.3911201869765686], [0.6017477630949866, 0.6017477630949866]], [[0.906940752375073, 0.906940752375073], [0.9532355856034859, 0.9532355856034859], [0.33923955132500233, 0.33923955132500233], [0.4185538375735909, 0.4185538375735909]], [[0.9986824054392485, 0.9986824054392485], [0.06032896866405729, 0.06032896866405729], [0.9876970751763091, 0.9876970751763091], [0.20779740986005732, 0.20779740986005732]], [[-0.11876256935925927, -0.11876256935925927], [-0.19032397102781556, -0.19032397102781556], [0.2420479329461963, 0.2420479329461963], [-0.4148155225831794, -0.4148155225831794]], [[0.7045232152825566, 0.7045232152825566], [0.022325835024472617, 0.022325835024472617], [-0.3922772100779879, -0.3922772100779879], [-0.9836001854611481, -0.9836001854611481]], [[0.16475469255318048, 0.16475469255318048], [-0.6697923512901962, -0.6697923512901962], [-0.9150886515939956, -0.9150886515939956], [-0.7551178900038119, -0.7551178900038119]], [[0.03129270552844392, 0.03129270552844392], [-0.5287207733771773, -0.5287207733771773], [-0.14359159300430854, -0.14359159300430854], [0.08992322244362319, 0.08992322244362319]]], "dtype": "float64"}, {"__nd__": [[-0.7300577916625816, -0.826554957882084, -0.8265830749627116], [0.9886558763970976, -0.12579187435095512, -0.029743376843294957], [-0.39328785950128814, -0.47022389194073366, -1.0134300800272527], [0.08520314695276245, 0.1297799104607051, -0.9882670262026468], [-0.521574588262846, -0.37205687807960414, -0.013001227219736141], [0.12206458349706871, 0.5628964841757262, 0.6822280095730638], [0.5342272561016675, 0.6707642657514294, -0.5832083451476577], [-0.27724588248980464, 0.5718049001350097, 0.1443305393622824]], "dtype": "float64"}, {"__nd__": [[0.0033095635577581237, -0.3498389883177601, -0.3286708204078675, 0.1292042167423317, 0.06038274628325358, -0.052336088420970005, -0.07300377803346687, -0.30513471812301596], [0.06562218552342211, 0.18286149039642888, 0.29029755881358, 0.008686444438718466, -0.03609721465571288, -0.19374673579680057, 0.2162409726763181, 0.30173227980493506], [-0.10839416733556281, 0.051399374423752156, 0.016028361637042375, -0.10143929790299464, -0.44149088020846033, -0.06470011234781561, 0.10738196831947841, -0.2551421630437266], [-0.1482946578635226, -0.5804569718554997, 0.2270745647398821, 0.0884372037418319, 0.008039544055119635, -0.04781339170945507, -0.11265681387778814, -0.013931109282220567], [-0.05485644819851254, -0.043903680295703656, -0.0059770168270632884, 0.02221216244798312, 0.27382328585120397, -0.2562258706939074, 0.04630994777915647, -0.11040779731441598], [-0.20143371855430037, -0.41863740882807454, 0.10728438741244031, -0.1750759689522504, -0.13429477577057328, 0.08129583552874216, -0.19330054529611698, 0.0028364050860458633], [-0.015004877330166328, 0.13997945768405873, 0.17688999897127056, -0.15500697156707247, 0.160367881979007, -0.018870144826306976, 0.019939980600717205, -0.039335221189098483], [0.13532956669010057, -0.21605654914863093, 0.044209070258950614, 0.20608269845246868, -0.051913997048859545, 0.35669514945806113, -0.1318587707406924, 0.03600872878103075]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-0.24630426446007098, -0.3143751552826488, -0.5628854967922629, -0.27503323455843603, -0.03111853655996072, -0.4065485880646405, 0.43619786973939706, 0.7551633138072489], [-0.7694480756992659, 0.4124216803896946, 0.0890300409181247, 0.11666035347030146, 1.2252496637864183, 0.5971623228973126, 0.1621549232151912, 0.08655707622317238]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 5, 0.03], "e": {"__tup__": [{"__nd__": [0.864683803602109, 0.5661885203617898, 0.4919578211343544], "dtype": "float64"}, {"__nd__": [[-0.735782442435454, -0.8211740377387846, -0.8138292631082803], [0.9882537490415668, -0.12326420093385594, -0.03319905025647641], [-0.39442924326681633, -0.4637792322089949, -1.0121814742539097], [0.0843590406415744, 0.13281859192297266, -0.9868605359442851], [-0.5222477744568266, -0.38311016135089804, -0.033775784239650275], [0.11805305771930005, 0.5650264873440457, 0.6763826042795539], [0.53670294778393, 0.662767083636403, -0.5919502580337496], [-0.27397267213788223, 0.5603779919086495, 0.13409887554155944]], "dtype": "float64"}, {"__nd__": [[0.012971423222287046, -0.35394506655693386, -0.31850418478758985, 0.1325321994435043, 0.06819849204647141, -0.06252098849922726, -0.07339471659303191, -0.30926424525101853], [0.06384130416200665, 0.18271011922924563, 0.28816852368154816, 0.009107529254730016, -0.03675010770706211, -0.19171991565011182, 0.21658710787865132, 0.30386796994976656], [-0.10695194449065232, 0.05091164080761324, 0.01792077910418268, -0.09933838592189466, -0.4407672129415038, -0.06638476000585472, 0.10865209685214708, -0.2558254149330647], [-0.14764892538651347, -0.5811319021569707, 0.2273640925470531, 0.0878823160506385, 0.00901749041982817, -0.04828676728234849, -0.11343971341869513, -0.014348128477629023], [-0.06865013134681788, -0.035644520790728364, -0.022992703219727687, 0.017116769762220943, 0.26179315761885386, -0.23938685207140603, 0.047281960015188945, -0.10734205127913501], [-0.20474528115178064, -0.41728297355815147, 0.10239452587335143, -0.17645597371709867, -0.13660685720221083, 0.08554579434319672, -0.19305559244368625, 0.00420229922041748], [-0.01916471910830207, 0.14220194347018045, 0.17166867108192985, -0.15669001821330192, 0.15739185173695838, -0.014081151374384857, 0.019800350177595936, -0.03842912517987357], [0.12907430098119727, -0.21439449445994685, 0.03738716943328992, 0.2061293473601567, -0.059020416960565196, 0.3651104770209387, -0.1300177651149069, 0.03984580342121364]], "dtype": "float64"}, {"__nd__": [0.002015504332036228, -0.0019454007796388974, 0.000138258962808288, -0.00010259115863704531, -0.0019975930166709482, -0.0007260509918298164, -0.0011743744067129911, -0.0028702706359661316], "dtype": "float64"}, {"__nd__": [[-0.22829057779148237, -0.30987883821125245, -0.5439197194881681, -0.2671412734038274, -0.031171309161890343, -0.4224327250926341, 0.4337465019620422, 0.7408729831752031], [-0.7484191239910553, 0.40782349053411804, 0.11171337581441923, 0.12711757294462048, 1.224137695559552, 0.5808405999277156, 0.1635529978098733, 0.08000756434682285]], "dtype": "float64"}, {"__nd__": [0.0007088098520654888, -0.0035844942383039067], "dtype": "float64"}, {"__nd__": [[0.9223519525538225, -0.9965703649643348, 0.8000380046101452, -0.7769619313494003, 0.8022905153665392, 0.01642038831768942, -0.938246225842452, 0.68136769866835], [-0.6430205815299072, 0.1769990569439693, -0.19161000308977047, -0.6473806463434028, 0.027212358169753096, -0.016185271774360886, -0.19428997098520692, 0.04824780069825782], [0.1619269660610522, -0.5032838704670703, 0.7237485516749608, 0.0898413887704074, 0.30469732046051556, -0.310056677103777, -0.08082556965987901, 0.37726329984978707], [-0.5746321530740557, -0.7462179570364121, -0.9021154924478821, -0.8477845845798895, 0.5782834513134262, 0.8312357556978851, -0.7692750694315162, 0.3527895957067412]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(tbptt_train, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")